In [4]:
import re
import pandas as pd

In [5]:
NEGWORDS = {
    "en": ["no", "not", "without", "absent", "negative for", "no evidence of"],
    "es": ["no hay", "sin", "sin signos de", "no se observa", "ausencia de"],
}
KEYWORDS = {
    "ACL": {
        "en": ["acl tear", "acl rupture", "anterior cruciate ligament tear",
               "acl injury", "torn acl"],
        "es": ["rotura del lca", "ligamento cruzado anterior roto",
               "lesión del ligamento cruzado anterior"],
    },
    "MCL": {
        "en": ["mcl tear", "mcl sprain", "medial collateral ligament tear",
               "medial collateral ligament injury"],
        "es": ["rotura del lcm", "ligamento colateral medial"],
    },
    "Medial Meniscus": {
        "en": ["medial meniscus tear", "medial meniscal tear", "tear of the medial meniscus"],
        "es": ["rotura de menisco interno", "rotura del menisco medial",
               "desgarro del menisco interno", "menisco interno roto"],
    },
    "Lateral Meniscus": {
        "en": ["lateral meniscus tear", "lateral meniscal tear", "tear of the lateral meniscus"],
        "es": ["rotura de menisco externo", "rotura del menisco lateral",
               "desgarro del menisco externo"],
    },
    "Medial OA": {
        "en": ["medial compartment osteoarthritis", "medial osteoarthritis",
               "medial joint space narrowing"],
        "es": ["artrosis femorotibial medial", "artrosis del compartimento medial"],
    },
    "Lateral OA": {
        "en": ["lateral compartment osteoarthritis", "lateral osteoarthritis",
               "lateral joint space narrowing"],
        "es": ["artrosis femorotibial lateral", "artrosis del compartimento lateral"],
    },
    "PF OA": {
        "en": ["patellofemoral osteoarthritis", "patellofemoral joint space narrowing",
               "patellofemoral arthrosis"],
        "es": ["artrosis patelofemoral", "artrosis femoropatelar"],
    },
    "Effusion": {
        "en": ["joint effusion", "knee effusion", "effusion is present", "effusion noted"],
        "es": ["derrame articular", "derrame", "derrame de la rodilla"],
    },
    "Synovitis": {
        "en": ["synovitis", "synovial thickening", "synovial hypertrophy"],
        "es": ["sinovitis", "engrosamiento sinovial"],
    },
    "Baker's": {
        "en": ["baker cyst", "baker's cyst", "popliteal cyst"],
        "es": ["quiste de baker", "quiste poplíteo"],
    },
    "Contusion": {
        "en": ["bone contusion", "bone bruise", "marrow edema", "osseous contusion"],
        "es": ["contusión ósea", "edema óseo", "contusión medular"],
    },
    "Fracture": {
        "en": ["fracture is seen", "fracture noted", "fracture is present", "acute fracture"],
        "es": ["fractura", "fractura aguda"],
    },
}


In [6]:
labels = list(KEYWORDS.keys())
allNegs = [word for words in NEGWORDS.values() for word in words]

In [7]:
def _is_negated(text: str, targetCharIdx: int, searchWindow: int = 5) -> bool:
    """
    Check if a target keyword is negated in the text.

    Args:
        text (str): The input text to search for negation.
        targetIdx (int): The index of the target keyword in the text.
        searchWindow (int): The number of words to look back for negation.

    Returns:
        bool: True if the target keyword is negated, False otherwise.
    """
    # Split the text into words
    # Everything up to targetCharIdx is considered for negation checking
    wordsBefore = re.findall(r'\w+', text[:targetCharIdx].lower())
    
    # Find the index of the target keyword in the list of words
    startWordIdx = max(0, len(wordsBefore) - searchWindow)
    
    for word in wordsBefore[startWordIdx:]:
        if word in allNegs:
            return True
    return False

In [8]:
test1 = "There is no evidence of an ACL tear."
test2 = "The patient has a medial meniscus tear."
test3 = "No signs of lateral compartment osteoarthritis were observed."
test4 = "The MRI shows a bone contusion but no fracture."
test5 = "There is synovitis present in the knee joint."

_is_negated(test1, 5, 3)

False

In [9]:
def labelReport(reportText: str) -> dict:
    text = (reportText or "").lower()
    result = {}

    for label, langKeywords in KEYWORDS.items():
        detPos = False
        detNeg = False

        for phrases in langKeywords.values():
            for phrase in phrases:

                # i.e . "acl tear" in "there is no evidence of an acl tear"
                # m would be a match object for "acl tear" and would have info about the match, including its start and end indices in the text
                for m in re.finditer(re.escape(phrase.lower()), text):
                    if _is_negated(text, m.start()):
                        detNeg = True
                    else:
                        detPos = True

                if detPos:
                    result[label] = 1
                elif detNeg:
                    result[label] = 0
                else:
                    result[label] = None
    return result


In [10]:
test = "the patient has a medial meniscus tear but no evidence of an ACL tear."

In [11]:
labelReport(test)

{'ACL': 0,
 'MCL': None,
 'Medial Meniscus': 1,
 'Lateral Meniscus': None,
 'Medial OA': None,
 'Lateral OA': None,
 'PF OA': None,
 'Effusion': None,
 'Synovitis': None,
 "Baker's": None,
 'Contusion': None,
 'Fracture': None}